#Introdution

**Description: ** Conditions are defined by Concepts from the Condition domain, which form a complex hierarchy. As a result, the same Person with the same disease may have multiple Condition records, which belong to the same hierarchical family. Most Condition records are mapped from diagnostic codes, but recorded signs, symptoms and summary descriptions also contribute to this table. Rule out diagnoses should not be recorded in this table, but in reality their negating nature is not always captured in the source data, and other precautions must be taken when when identifying Persons who should suffer from the recorded Condition. Record all conditions as they exist in the source data. Any decisions about diagnosis/phenotype definitions would be done through cohort specifications. These cohorts can be housed in the COHORT table. Conditions span a time interval from start to end, but are typically recorded as single snapshot records with no end date. The reason is twofold: (i) At the time of the recording the duration is not known and later not recorded, and (ii) the Persons typically cease interacting with the healthcare system when they feel better, which leads to incomplete capture of resolved Conditions. The CONDITION_ERA table addresses this issue. Family history and past diagnoses (‘history of’) are not recorded in this table. Instead, they are listed in the OBSERVATION table. Codes written in the process of establishing the diagnosis, such as ‘question of’ of and ‘rule out’, should not represented here. Instead, they should be recorded in the OBSERVATION table, if they are used for analyses. However, this information is not always available.

In [4]:
import pandas as pd

In [11]:
condition_occurrence = pd.read_csv("../data/condition_occurrence_202611061452.csv")

In [18]:
visit_occurrence = pd.read_csv("../data/visit_occurrence_202611061448.csv")
visit_occurrence["visit_start_datetime"] = pd.to_datetime(visit_occurrence["visit_start_datetime"])

In [ ]:
#Rows and columns values
rows, column = (condition_occurrence.shape)
print(f"rows: {rows} \ncolumn {column}")

total_visit = len(condition_occurrence)
print(f"total_visit: {total_visit}")


KeyboardInterrupt



In [ ]:
#Null values
list_columns = condition_occurrence.columns.to_list()

for i in list_columns:
    num_nulls = condition_occurrence[i].isnull().sum()
    
    if num_nulls > 0:
        print(f"The column '{i}' HAS nulls ({num_nulls} in total)")
    else:
        print(f"The column '{i}' HAS NO nulls")

The column 'condition_occurrence_id' HAS NO nulls
The column 'person_id' HAS NO nulls
The column 'visit_occurrence_id' HAS nulls (823429 in total)
The column 'condition_concept_id' HAS NO nulls
The column 'condition_start_datetime' HAS NO nulls
The column 'condition_type_concept_id' HAS NO nulls


In [ ]:
#Duplicate values
list_columns = condition_occurrence.columns.to_list()

for i in list_columns:
    num_duplicate = condition_occurrence[i].duplicated().sum()

    if num_duplicate > 0:
        print(f"The column {i} has duplicate values ({num_duplicate} in total)")
    else:
        print(f"The column {i} has no duplicate")

#Duplicate total per rows
duplicate_records = condition_occurrence.duplicated().sum()
print("Total of total duplicate per rows: ", duplicate_records)

The column condition_occurrence_id has no duplicate
The column person_id has duplicate values (38429668 in total)
The column visit_occurrence_id has duplicate values (38070059 in total)
The column condition_concept_id has duplicate values (38631918 in total)
The column condition_start_datetime has duplicate values (32615155 in total)
The column condition_type_concept_id has duplicate values (38636062 in total)
Total of total duplicate per rows:  0


In [ ]:
#Check if the visit_occurrence_id exist in visit_occurrence
values_exist = (visit_occurrence["visit_occurrence_id"].isin(condition_occurrence["visit_occurrence_id"])).sum()
values_not_exist = (~visit_occurrence["visit_occurrence_id"].isin(condition_occurrence["visit_occurrence_id"])).sum()
print(f"Numbers of IDs that don't exist in visit_occurrence: {values_exist}")
print(f"Numbers of IDs that exist in visit_occurrence: {values_not_exist}")


Numbers of IDs that don't exist in visit_occurrence: 565827
Numbers of IDs that exist in visit_occurrence: 1140174


In [ ]:
#Conditions regarding visits
condition_per_visit = condition_occurrence["visit_occurrence_id"].value_counts()
condition_distribution = condition_per_visit.value_counts().sort_index()
print(condition_distribution)

#Maximun condition
print(f"The higher maximun condition is: {condition_per_visit.max()}")
print(f"The value most apper is: {condition_per_visit.idxmax()}")

exits_id = condition_occurrence["visit_occurrence_id"].isin([condition_per_visit.idxmax()]).any()
if exits_id:
  print(f"The ID {exits_id} is in the table")
else:
  print(f"El ID {exits_id} is in the table")

count
1        23742
2        94824
3        28168
4        47668
5         4293
         ...  
24003        1
26418        1
31850        1
34590        1
47286        1
Name: count, Length: 2675, dtype: int64
The higher maximun condition is: 47286
The value most apper is: 2415586.0
El ID True SÍ se encuentra en la tabla.


In [19]:
condition_occurrence["condition_start_datetime"] = pd.to_datetime(condition_occurrence["condition_start_datetime"])
max_date = condition_occurrence["condition_start_datetime"].max()
min_date = condition_occurrence["condition_start_datetime"].min()
print(f"The last date is: {max_date}\nThe first date is: {min_date}")

amount = (condition_occurrence["condition_start_datetime"] > visit_occurrence["visit_start_datetime"].max()).sum()
print("Subsequent records: ", amount)

The last date is: 2033-05-29 09:38:26
The first date is: 2023-01-01 00:01:50
Subsequent records:  82


**Temporal Data Quality – CONDITION_OCCURRENCE**

The `condition_start_datetime` column ranges from January 1, 2023, to May 29, 2033. A total of 82 records have timestamps later than the latest recorded visit date (August 13, 2026), representing less than 0.001% of the dataset. These records indicate potential temporal inconsistencies that should be investigated or handled during data preprocessing.